# Step 3 — Accuracy comparison: pipeline output vs ground truth

## Prerequisites

- `output_box_fixing/corrected_boxes.json` — the 79 corrected bounding boxes from step 1
- `output_ground_truth/ground_truth.json` — the manually typed field values from step 2

## What this notebook does

1. Runs the Gemini extraction pipeline on page 200 using the corrected bounding boxes
2. Loads the hand-typed ground truth
3. Compares every field of every entry — pipeline output vs ground truth
4. Reports per-field accuracy (which fields does the pipeline get right?)
5. Saves the mismatches so each specific error can be inspected

This produces the first real accuracy number for the project — not a match score (which only measures faithfulness to OCR text), but a field-level correctness score measured against human-verified data.

## 0. Setup

In [1]:
import os
import re
import json
import time
from pathlib import Path
from typing import Optional, Literal, List, Tuple, Dict

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from pydantic import BaseModel
from google import genai
from google.genai import types

# API
KEY_FILE = Path("api_key.txt")
API_KEY = (
    KEY_FILE.read_text().strip()
    if KEY_FILE.exists()
    else os.getenv("GEMINI_API_KEY", "")
)
assert API_KEY, "API key required in api_key.txt or GEMINI_API_KEY"

MODEL_NAME = "gemini-2.5-flash"
client = genai.Client(api_key=API_KEY)

# Paths
BOXES_PATH = Path("output_box_fixing") / "corrected_boxes.json"
GT_PATH = Path("output_ground_truth") / "ground_truth.json"
OUTPUT_DIR = Path("output_step3_accuracy")
OUTPUT_DIR.mkdir(exist_ok=True)

assert BOXES_PATH.exists(), f"Missing corrected boxes: {BOXES_PATH}"
assert GT_PATH.exists(), f"Missing ground truth: {GT_PATH}"

print("Setup OK")
print(f"  Boxes: {BOXES_PATH}")
print(f"  Ground truth: {GT_PATH}")

Setup OK
  Boxes: output_box_fixing\corrected_boxes.json
  Ground truth: output_ground_truth\ground_truth.json


---
## 1. Load the corrected boxes and page image

In [2]:
box_data = json.loads(BOXES_PATH.read_text())
boxes = box_data["boxes"]

DATA_DIR = Path(r"C:\Users\ABHIRAMI.K\Documents\RICE\Summer 2026\Fondren Internship\Data\image")
PAGE_PATH = DATA_DIR / "1900-1901 (page 200).png"

page_img = cv2.imread(str(PAGE_PATH))
assert page_img is not None, f"Page image not found: {PAGE_PATH}"

print(f"Loaded {len(boxes)} corrected boxes")
print(f"Page: {page_img.shape[1]}x{page_img.shape[0]}")

Loaded 79 corrected boxes
Page: 1806x2478


---
## 2. OCR each column with Gemini

The corrected boxes define where each entry is, but the OCR still needs to read the text. The approach here: OCR each column as a single image (same as Week 5), then pair the OCR lines to boxes by vertical position.

This avoids the problems from earlier iterations where Gemini was asked to produce both boxes and text (unreliable boxes) or where boxes and text were paired by simple index (fragile when counts disagree).

In [3]:
OCR_PROMPT = """Transcribe ONE column from a page of the 1900-1901 Houston city directory.

Rules:
- One directory entry per output line
- Join continuation lines into their parent entry
- Preserve abbreviations exactly: r., h., bds, rms, hhldr, (c), (col), wid, propr, clk, lab, engr, wks
- Preserve commas and periods
- Do NOT expand abbreviations or correct spelling
- Write [?] for unclear characters
- Skip page headers, page numbers, guide words, and advertisement text

Return only the transcribed entries, one per line, no commentary."""


def ocr_column(column_img):
    """Transcribe one column image using Gemini Vision."""
    rgb = cv2.cvtColor(column_img, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=[OCR_PROMPT, pil],
    )
    return (response.text or "").strip()


# Detect columns using the same OpenCV approach as before
def detect_columns(img):
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    sm = int(w * 0.08); tm, bm = int(h * 0.08), int(h * 0.92)
    core = binary[tm:bm, sm:w-sm]
    vproj = core.sum(axis=0)
    win = max(5, core.shape[1] // 60)
    smoothed = np.convolve(vproj, np.ones(win)/win, mode="same")
    ms, me = len(smoothed)//3, 2*len(smoothed)//3
    gap = sm + ms + int(np.argmin(smoothed[ms:me]))
    buf = int(w * 0.01)
    return (sm, tm, gap-buf, bm), (gap+buf, tm, w-sm, bm)


left_col, right_col = detect_columns(page_img)

print("Running OCR on left column...")
left_text = ocr_column(page_img[left_col[1]:left_col[3], left_col[0]:left_col[2]])
left_lines = [ln.strip() for ln in left_text.split("\n") if ln.strip()]

print("Running OCR on right column...")
right_text = ocr_column(page_img[right_col[1]:right_col[3], right_col[0]:right_col[2]])
right_lines = [ln.strip() for ln in right_text.split("\n") if ln.strip()]

print(f"  Left column:  {len(left_lines)} OCR lines")
print(f"  Right column: {len(right_lines)} OCR lines")

# Pair OCR lines to boxes by column
left_boxes = sorted([b for b in boxes if b["col"] == "L"], key=lambda b: b["y1"])
right_boxes = sorted([b for b in boxes if b["col"] == "R"], key=lambda b: b["y1"])

print(f"  Left boxes:   {len(left_boxes)}")
print(f"  Right boxes:  {len(right_boxes)}")

Running OCR on left column...
Running OCR on right column...
  Left column:  39 OCR lines
  Right column: 42 OCR lines
  Left boxes:   35
  Right boxes:  40


---
## 3. Parse each OCR line into structured fields

The same schema-enforced extraction with the updated fields (workplace_address, ownership_type, etc.). Corrections from the ground truth can optionally be injected as few-shot examples.

In [4]:
class DirectoryEntry(BaseModel):
    """Schema matching the ground-truth fields from step 2."""
    is_directory_entry: bool
    not_entry_reason: Optional[str] = None
    entry_type: Literal["person", "business", "institution", "cross_reference", "unclear"] = "person"

    last_name: Optional[str] = None
    first_name: Optional[str] = None
    business_name: Optional[str] = None

    racial_marker_raw: Optional[str] = None

    occupation_raw: Optional[str] = None
    employer: Optional[str] = None
    workplace_address: Optional[str] = None

    residence_raw: Optional[str] = None
    boarding_raw: Optional[str] = None
    rooms_raw: Optional[str] = None
    residence_qualifier: Optional[str] = None
    ownership_type: Optional[Literal["home", "householder"]] = None

    spouse_or_relation_raw: Optional[str] = None
    notes: Optional[str] = None


PARSE_PROMPT = """Parse this 1900-1901 Houston directory entry into structured fields.

Entry text: {raw_text}

Rules:
1. If this is NOT a real listing (ad, header, page number, slogan), set is_directory_entry=false and leave all fields null.

2. Split occupation and employer separately:
   - "wks S. P. shops" -> occupation_raw="wks", employer="S. P. shops"
   - "engr H. & T. C. R. R." -> occupation_raw="engr", employer="H. & T. C. R. R."

3. Workplace address is separate from residence:
   - "barber, shop 1102 McKee, r. 904 McKee" -> workplace_address="shop 1102 McKee", residence_raw="r. 904 McKee"
   - "physician, 602½ Main, r. 1703 Preston ave." -> workplace_address="602½ Main", residence_raw="r. 1703 Preston ave."

4. ownership_type: "h." -> "home", "hhldr" -> "householder", otherwise null

5. Widow status, phone numbers, cross-references, and other remarks go in notes.

6. For cross-references like "See also Haden, Heyden" -> entry_type="cross_reference", put text in notes.

7. Copy abbreviations exactly. Do not expand. Use null for missing fields.
"""


def parse_entry(raw_text: str) -> DirectoryEntry:
    """Parse one OCR line into structured fields."""
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=PARSE_PROMPT.format(raw_text=raw_text),
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=DirectoryEntry,
        ),
    )
    return DirectoryEntry.model_validate_json(response.text)

---
## 4. Run the pipeline on all OCR lines

In [5]:
def process_column(ocr_lines, column_boxes, col_label):
    """Parse all OCR lines for one column. Returns a list of dicts."""
    results = []
    for i, raw_line in enumerate(ocr_lines):
        box_id = column_boxes[i]["id"] if i < len(column_boxes) else -1
        try:
            entry = parse_entry(raw_line)
        except Exception as e:
            print(f"  [{col_label}-{i}] parse failed: {e}")
            continue
        results.append({
            "box_id":             box_id,
            "column":             col_label,
            "line_index":         i,
            "raw_text":           raw_line,
            "is_directory_entry": entry.is_directory_entry,
            "entry_type":         entry.entry_type,
            "last_name":          entry.last_name,
            "first_name":         entry.first_name,
            "business_name":      entry.business_name,
            "race":               entry.racial_marker_raw,
            "occupation":         entry.occupation_raw,
            "employer":           entry.employer,
            "workplace_address":  entry.workplace_address,
            "residence":          entry.residence_raw,
            "boarding":           entry.boarding_raw,
            "rooms":              entry.rooms_raw,
            "residence_qualifier": entry.residence_qualifier,
            "ownership":          entry.ownership_type,
            "notes":              entry.notes,
        })
    return results


print("Parsing left column...")
left_results = process_column(left_lines, left_boxes, "L")
print(f"  {len(left_results)} entries parsed")

print("Parsing right column...")
right_results = process_column(right_lines, right_boxes, "R")
print(f"  {len(right_results)} entries parsed")

all_results = left_results + right_results
pipeline_df = pd.DataFrame(all_results)
pipeline_df.to_csv(OUTPUT_DIR / "pipeline_output.csv", index=False)
print(f"\nTotal pipeline output: {len(pipeline_df)} entries")
print(f"Saved: {OUTPUT_DIR / 'pipeline_output.csv'}")

Parsing left column...
  39 entries parsed
Parsing right column...
  42 entries parsed

Total pipeline output: 81 entries
Saved: output_step3_accuracy\pipeline_output.csv


---
## 5. Load the ground truth and align entries

The ground truth JSON has entries keyed by `box_id`. The pipeline output has entries keyed by `box_id` too (via the corrected boxes). Matching them is straightforward: for each box, compare the pipeline's extraction against the ground truth's fields.

For boxes with multiple entries (where the user clicked "+ Add entry"), each ground-truth entry is compared against the pipeline entry from the same position within that box.

In [6]:
# Load ground truth
gt_data = json.loads(GT_PATH.read_text())
gt_boxes = gt_data["boxes"]

# Build a lookup: box_id -> list of ground truth entries
gt_by_box: Dict[int, List[dict]] = {}
for gb in gt_boxes:
    if gb["skipped"]:
        continue
    gt_by_box[gb["box_id"]] = gb["entries"]

# Build a lookup: box_id -> list of pipeline entries
pipe_by_box: Dict[int, List[dict]] = {}
for _, row in pipeline_df.iterrows():
    bid = row["box_id"]
    if bid not in pipe_by_box:
        pipe_by_box[bid] = []
    pipe_by_box[bid].append(row.to_dict())

print(f"Ground truth boxes with entries: {len(gt_by_box)}")
print(f"Pipeline boxes with entries:     {len(pipe_by_box)}")
print(f"Boxes in common:                 {len(set(gt_by_box) & set(pipe_by_box))}")

Ground truth boxes with entries: 79
Pipeline boxes with entries:     76
Boxes in common:                 75


---
## 6. Per-field accuracy comparison

For each entry, each field is compared case-insensitively with whitespace stripped. The field mapping between the ground truth schema and the pipeline output schema is explicit below.

The output is a table showing, for every field: how many entries were correct, how many were wrong, and the accuracy percentage.

In [7]:
# Mapping from ground truth field names to pipeline output field names
FIELD_MAP = {
    "last_name":           "last_name",
    "first_name":          "first_name",
    "business_name":       "business_name",
    "race":                "race",
    "occupation":          "occupation",
    "employer":            "employer",
    "workplace_address":   "workplace_address",
    "residence":           "residence",
    "boarding":            "boarding",
    "rooms":               "rooms",
    "residence_qualifier": "residence_qualifier",
    "ownership":           "ownership",
    "entry_type":          "entry_type",
    "notes":               "notes",
}


def normalize_value(v) -> str:
    """Normalize a field value for comparison: lowercase, strip whitespace,
    collapse multiple spaces, strip trailing periods."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    s = str(v).strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = s.rstrip(".")
    return s


def compare_entries(gt_entry: dict, pipe_entry: dict) -> Dict[str, dict]:
    """Compare one ground-truth entry against one pipeline entry.
    Returns a dict of field_name -> {gt, pipe, match}."""
    results = {}
    for gt_field, pipe_field in FIELD_MAP.items():
        gt_val = normalize_value(gt_entry.get(gt_field))
        pipe_val = normalize_value(pipe_entry.get(pipe_field))
        results[gt_field] = {
            "gt":    gt_val,
            "pipe":  pipe_val,
            "match": gt_val == pipe_val,
        }
    return results


# Compare all aligned entries
all_comparisons = []
for box_id in sorted(gt_by_box.keys()):
    gt_entries = gt_by_box[box_id]
    pipe_entries = pipe_by_box.get(box_id, [])

    for entry_idx, gt_entry in enumerate(gt_entries):
        pipe_entry = pipe_entries[entry_idx] if entry_idx < len(pipe_entries) else {}
        comparison = compare_entries(gt_entry, pipe_entry)
        for field_name, result in comparison.items():
            all_comparisons.append({
                "box_id":     box_id,
                "entry_idx":  entry_idx,
                "field":      field_name,
                "gt_value":   result["gt"],
                "pipe_value": result["pipe"],
                "match":      result["match"],
            })

comp_df = pd.DataFrame(all_comparisons)
comp_df.to_csv(OUTPUT_DIR / "field_comparisons.csv", index=False)

# Per-field accuracy summary
print("=" * 65)
print("PER-FIELD ACCURACY — PIPELINE vs GROUND TRUTH")
print("=" * 65)

summary_rows = []
for field_name in FIELD_MAP.keys():
    field_df = comp_df[comp_df["field"] == field_name]
    # Only count entries where at least one side has a value
    # (comparing blank-to-blank is meaningless — both correctly identified absence)
    has_value = field_df[(field_df["gt_value"] != "") | (field_df["pipe_value"] != "")]
    if len(has_value) == 0:
        continue
    correct = has_value["match"].sum()
    total = len(has_value)
    accuracy = correct / total if total else 0.0
    summary_rows.append({
        "field":      field_name,
        "correct":    int(correct),
        "wrong":      total - int(correct),
        "total":      total,
        "accuracy":   round(accuracy, 3),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# Overall
total_fields = summary_df["total"].sum()
total_correct = summary_df["correct"].sum()
overall_acc = total_correct / total_fields if total_fields else 0.0
print(f"\nOverall field-level accuracy: {total_correct}/{total_fields} = {overall_acc:.1%}")
print("=" * 65)

summary_df.to_csv(OUTPUT_DIR / "per_field_accuracy.csv", index=False)
print(f"\nSaved: {OUTPUT_DIR / 'per_field_accuracy.csv'}")

PER-FIELD ACCURACY — PIPELINE vs GROUND TRUTH
              field  correct  wrong  total  accuracy
          last_name       70      9     79     0.886
         first_name       61     17     78     0.782
      business_name        2      0      2     1.000
               race       27     11     38     0.711
         occupation       47     16     63     0.746
           employer       28     14     42     0.667
  workplace_address        4      2      6     0.667
          residence       30     43     73     0.411
           boarding        3      8     11     0.273
              rooms        0      7      7     0.000
residence_qualifier        1      3      4     0.250
          ownership        9      6     15     0.600
         entry_type       74      7     81     0.914
              notes        4      9     13     0.308

Overall field-level accuracy: 360/512 = 70.3%

Saved: output_step3_accuracy\per_field_accuracy.csv


---
## 7. Inspect the mismatches

The entries below are the ones where the pipeline disagrees with the ground truth. These are the specific errors to understand — are they OCR errors, parsing errors, or schema interpretation differences?

In [8]:
mismatches = comp_df[~comp_df["match"] &
                    ((comp_df["gt_value"] != "") | (comp_df["pipe_value"] != ""))].copy()

print(f"Total mismatches: {len(mismatches)}")
print()

if len(mismatches):
    # Group by field to see which fields have the most errors
    print("Mismatches by field:")
    for field, group in mismatches.groupby("field"):
        print(f"\n  {field}: {len(group)} errors")
        for _, row in group.head(3).iterrows():
            print(f"    box {row['box_id']}: GT='{row['gt_value']}' vs PIPE='{row['pipe_value']}'")

    mismatches.to_csv(OUTPUT_DIR / "mismatches.csv", index=False)
    print(f"\nFull mismatch list: {OUTPUT_DIR / 'mismatches.csv'}")
else:
    print("No mismatches found — pipeline matches ground truth perfectly.")

Total mismatches: 152

Mismatches by field:

  boarding: 8 errors
    box 3: GT='bds 803 main' vs PIPE='bds'
    box 6: GT='bds 1404 polk ave' vs PIPE='bds'
    box 13: GT='bds germania house' vs PIPE='germania house'

  employer: 14 errors
    box 8: GT='auditorâ€™s office h. & t. c. r. r' vs PIPE='h. & t. c. r. r'
    box 32: GT='si packard tro ldy' vs PIPE='si packard troy ldy'
    box 35: GT='h. h. franks' vs PIPE=''

  entry_type: 7 errors
    box 35: GT='person' vs PIPE=''
    box 36: GT='person' vs PIPE=''
    box 37: GT='person' vs PIPE=''

  first_name: 17 errors
    box 16: GT='catherine' vs PIPE='catharine'
    box 35: GT='lela' vs PIPE=''
    box 36: GT='lewis e' vs PIPE=''

  last_name: 9 errors
    box 35: GT='hawkins' vs PIPE=''
    box 36: GT='hawkins' vs PIPE=''
    box 37: GT='hawkins' vs PIPE=''

  notes: 9 errors
    box 4: GT='edward t. hatfield, earl p. hopkins' vs PIPE='(edward t. hatfield, earl p. hopkins)'
    box 7: GT='wid wm. l' vs PIPE='(wid wm. l.)'
    bo

---
## 8. Summary and next steps

After running this notebook, the key output files are:

| File | Purpose |
|---|---|
| `pipeline_output.csv` | Every field the pipeline extracted |
| `field_comparisons.csv` | Every field comparison (GT vs pipeline, match/no-match) |
| `per_field_accuracy.csv` | Per-field accuracy summary |
| `mismatches.csv` | Only the fields where pipeline ≠ ground truth |

## How to interpret the results

- **Per-field accuracy above 90%** → the pipeline handles that field well; remaining errors are edge cases.
- **Per-field accuracy between 70-90%** → systematic issue; examine the mismatches to find the pattern.
- **Per-field accuracy below 70%** → fundamental problem with how the pipeline handles that field type.

## What comes after this (step 4)

The mismatches from this comparison become few-shot examples in the extraction prompt. For every entry where the pipeline got a field wrong, the correct answer (from the ground truth) is added to the prompt as an example. This is the in-context learning loop:

```
Step 2 (ground truth) + Step 3 (mismatches) → few-shot examples → re-run pipeline → measure again
```

The next notebook (step 4) will implement this loop automatically.

